In [ ]:
# import yfinance as yf

# # สร้าง Ticker Object
# sp500 = yf.Ticker("^GSPC")

# # ดึงข้อมูลประวัติราคา (Historical Data)
# # period: ระยะเวลาถอยหลัง (1d, 5d, 1mo, 1y, 5y, max)
# # interval: ความละเอียดของข้อมูล (1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo)
# df = sp500.history(period="max", interval="1d")

# print(df.head())


In [2]:
import yfinance as yf

# สร้าง Ticker Object
xau = yf.Ticker("^xau")

# ดึงข้อมูลประวัติราคา (Historical Data)
# period: ระยะเวลาถอยหลัง (1d, 5d, 1mo, 1y, 5y, max)
# interval: ความละเอียดของข้อมูล (1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo)
df = xau.history(period="max", interval="1d")

# print(df.head())


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
df.describe()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,10680.000000,10680.000000,10680.000000,10680.000000,1.068000e+04,10680.0,10680.0
mean,112.292628,113.736083,110.803253,112.286999,3.987677e+04,0.0,0.0
std,49.367506,50.098533,48.567320,49.399314,6.238374e+05,0.0,0.0
min,39.520000,40.430000,38.360001,38.840000,0.000000e+00,0.0,0.0
25%,80.559998,81.547503,79.667500,80.599998,0.000000e+00,0.0,0.0
50%,100.494999,101.785000,99.290001,100.584999,1.890000e+04,0.0,0.0
75%,134.110001,135.612495,132.210007,133.782497,5.320000e+04,0.0,0.0
max,468.739990,472.079987,463.529999,470.369995,4.926950e+07,0.0,0.0


In [4]:
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

In [13]:
X, y = df[['Low', 'High', 'Volume']], df[['Open', 'Close']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    objective='reg:squarederror' # ระบุว่าเป็นงาน Regression
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# วัดผลแยกตามคอลัมน์ (Open และ High)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")   # ค่าเฉลี่ยที่ทายผิดไป (ดอลลาร์)
print(f"RMSE: {rmse:.2f}") # ค่าความผิดพลาดเฉลี่ยแบบ Root Mean Square
print(f"R2 Score: {r2:.4f}") # ค่าความสัมพันธ์ (เข้าใกล้ 1 คือดีมาก)

MAE: 1.18
RMSE: 2.66
R2 Score: 0.9971


In [15]:
import mlflow
from sklearn.metrics import mean_squared_error
import numpy as np

with mlflow.start_run(run_name="Manual_Tracking_Run"):
    # --- ส่วนการเทรนโมเดล ---
    model = XGBRegressor(n_estimators=100, learning_rate=0.1)
    model.fit(X_train, y_train)
    
    # --- ส่วนการคำนวณ Metrics ---
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    # --- การบันทึกข้อมูลลง MLflow ---
    # บันทึก Parameter ที่เราสนใจ
    mlflow.log_param("model_type", "XGBRegressor")
    mlflow.log_param("n_estimators", 100)
    
    # บันทึก Metric (สำคัญมากสำหรับการทำกราฟเปรียบเทียบแต่ละสัปดาห์)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    # บันทึกตัวโมเดลไว้ใช้งานต่อ (Model Artifact)
    mlflow.xgboost.log_model(model, "model")
    
    print(f"Run finished. RMSE: {rmse} logged to MLflow.")

2026/05/09 22:36:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run finished. RMSE: 2.6684991580189146 logged to MLflow.
